# The K-Suiter: patient-specific EEG channel ranking

Given a patient and the `K` saved by K-Finder, this notebook ranks and returns the `K` EEG channels that best suit that patient's seizure-forecasting history. The latest seizure/control pair is reserved for a final untouched evaluation.

The ranking uses expanding chronological validation: models learn from earlier seizure episodes and validate on later episodes. At each step, every remaining channel is tested alongside the channels already selected. The candidate with the highest value is added:

$$\text{value} = 2\times\text{sensitivity} - 0.10\times\text{false alarms/hour} - 0.25\times\text{time in warning} + 0.20\times\text{AUROC}$$

The utility rewards seizure capture and AUROC while penalizing false alarms and time in warning, matching the rolling forecast alarm policy. Selection is repeated across expanding historical prefixes and records channel stability.

> **Research only:** this ranking has not been clinically validated and must not directly control patient care.

## 1. Inputs

Set the patient identifier. The channel count `K` is loaded automatically from the versioned result written by `k-finder.ipynb`, so it is never copied from a displayed notebook output. Siena identifiers look like `PN00`, `PN06`, and `PN10`.

In [1]:
PATIENT_ID = "PN00"
# K is read from results/sensor_count_step1.json, written by k-finder.ipynb.
FORCE_REBUILD_FEATURES = False

## 2. Setup

The notebook can be launched from either the scripts directory or the repository root. Feature extraction is cached; the first run for a patient may take several minutes.

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "k_suiter.py").exists():
    candidates = list(NOTEBOOK_DIR.rglob("k_suiter.py"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Run this notebook from the scripts directory or repository root."
        )
    NOTEBOOK_DIR = candidates[0].parent
sys.path.insert(0, str(NOTEBOOK_DIR))

import rolling_seizure_forecasting as rsf
import personalized_channels_workflow as pc
from k_suiter import KSuiter, load_k_finder_result

paths = pc.personalized_paths(NOTEBOOK_DIR)
forecast_config = rsf.ForecastConfig(test_fraction=0.20, max_iter=60)
k_finder_result = load_k_finder_result(paths["project"])
K = k_finder_result.k
suiter_config = pc.PersonalizedConfig(
    k=K,
    patient_ids=(PATIENT_ID,),
    swap_refinement=False,
    force_rebuild_features=FORCE_REBUILD_FEATURES,
)
suiter_config.validate()
print(f"Patient={PATIENT_ID}; K={K} (from {k_finder_result.source_path.name})")

Patient=PN00; K=2 (from sensor_count_step1.json)


## 3. Load one patient's channel-local features

Eligibility requires at least two usable historical seizure events and at least `K` consistently available channels. Only channel-local features are used, so an excluded electrode cannot leak information into a selected electrode.

In [3]:
manifest = pc.load_manifest(paths, forecast_config)
available_patients = sorted(manifest["patient_id"].astype(str).unique())
if PATIENT_ID not in available_patients:
    raise ValueError(
        f"Unknown patient {PATIENT_ID!r}. Available patients: {available_patients}"
    )

patient_manifest = manifest.loc[
    manifest["patient_id"].astype(str).eq(PATIENT_ID)
].copy()
n_events = patient_manifest.loc[
    patient_manifest["episode_type"].eq("preictal"), "source_event_id"
].nunique()
if n_events < 2:
    raise ValueError(
        f"{PATIENT_ID} has {n_events} usable seizure event(s); at least 2 are required."
    )

patient = pc.build_patient_feature_data(
    patient_manifest,
    paths["feature_cache"],
    forecast_config,
    force=FORCE_REBUILD_FEATURES,
)
patient_overview = pd.DataFrame(
    {
        "patient_id": [patient.patient_id],
        "usable_seizure_events": [n_events],
        "available_channels": [len(patient.channel_names)],
        "landmark_rows": [len(patient.frame)],
    }
)
display(patient_overview)

PN00 [1/25] PN00_S01_preictal
PN00 [2/25] PN00_S02_preictal
PN00 [3/25] PN00_S03_preictal
PN00 [4/25] PN00_S04_preictal
PN00 [5/25] PN00_S05_preictal
PN00 [6/25] PN00_interictal_01
PN00 [7/25] PN00_interictal_02
PN00 [8/25] PN00_interictal_03
PN00 [9/25] PN00_interictal_04
PN00 [10/25] PN00_interictal_05
PN00 [11/25] PN00_interictal_06
PN00 [12/25] PN00_interictal_07
PN00 [13/25] PN00_interictal_08
PN00 [14/25] PN00_interictal_09
PN00 [15/25] PN00_interictal_10
PN00 [16/25] PN00_interictal_11
PN00 [17/25] PN00_interictal_12
PN00 [18/25] PN00_interictal_13
PN00 [19/25] PN00_interictal_14
PN00 [20/25] PN00_interictal_15
PN00 [21/25] PN00_interictal_16
PN00 [22/25] PN00_interictal_17
PN00 [23/25] PN00_interictal_18
PN00 [24/25] PN00_interictal_19
PN00 [25/25] PN00_interictal_20


,patient_id,usable_seizure_events,available_channels,landmark_rows
0,PN00,5,29,1500


## 4. Run the K-Suiter

The first result is the requested simple output: the ordered canonical channel names. The table adds audit details, and the notebook writes both a JSON handoff file (`included_eeg_channels`) and a ranking CSV under `results/k_suiter`. `marginal_value` measures the change in the complete selected set's value after adding that channel; it need not always be positive.

In [4]:
suiter = KSuiter(config=suiter_config)
selected_channels = suiter.recommend_from_k_finder(patient, paths["project"])
recommendation_json, ranking_csv = suiter.save_recommendation(
    paths["project"] / "results" / "k_suiter"
)

print(f"Top {K} channels for {PATIENT_ID}: {selected_channels}")
print(f"Saved model-ready channels: {recommendation_json}")
print(f"Saved ranking audit: {ranking_csv}")
display(suiter.ranking_.style.format(
    {
        "validation_alarm_utility": "{:.4f}",
        "stability_frequency": "{:.0%}",
        "validation_sensitivity": "{:.3f}",
        "validation_false_alarms_per_hour": "{:.3f}",
        "validation_auprc": "{:.4f}",
        "validation_brier": "{:.4f}",
    }
))

Top 2 channels for PN00: ['C3', 'F9']
Saved model-ready channels: /Users/advaitghosh/Documents/cosmos/26-the-optimizers-analysis/final_project/results/k_suiter/k_suiter_PN00_k2_recommendation.json
Saved ranking audit: /Users/advaitghosh/Documents/cosmos/26-the-optimizers-analysis/final_project/results/k_suiter/k_suiter_PN00_k2_ranking.csv


,rank,step,channel,channels,selection_score,validation_alarm_utility,stability_frequency,validation_sensitivity,validation_false_alarms_per_hour,validation_time_in_warning,validation_auroc,validation_auprc,validation_brier
0,1,1,C3,C3,1.790005,1.7733,33%,1.000,3.000,0.317778,0.763912,0.3218,0.2333
1,2,2,F9,"C3, F9",1.417120,1.4005,33%,1.000,6.000,0.424444,0.532824,0.1957,0.4142


## 5. Interpretation

- Rank 1 is the strongest channel when used alone.
- Each later channel is the best addition conditional on the channels above it.
- Validation is chronological and patient-specific; results should not be interpreted as a universal electrode ranking.
- A high validation score on a patient with few events is uncertain. Report the event count with every ranking.
- The selected channels should be evaluated on a later untouched episode before making research performance claims.